<a href="https://colab.research.google.com/github/borhanur-rahman/Dengu_Biomarker_Discovery/blob/main/Dengue_research_for_Bangladeshi_People.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1: Mount Google Drive & Install Dependencies
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q diptest matplotlib reportlab fpdf2

print("✓ All packages installed")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.7/222.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 12.7 MB/s eta 0:00:00
✓ All packages installed


In [4]:
!pip install -q diptest
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q tqdm

print("✓ All packages installed")

✓ All packages installed


In [5]:
# ============================================================
# CELL 2: Configuration
# ============================================================
from pathlib import Path

# ── Paths ───────────────────────────────────────────────────
EXPR_FILE = "/content/drive/MyDrive/Compendium_Dataset/compendium_expression.csv.gz"
META_FILE = "/content/drive/MyDrive/Compendium_Dataset/compendium_metadata.csv"

OUT_DIR = Path("/content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Filtering parameters ────────────────────────────────────
HARTIGAN_DIP_ALPHA  = 0.05
BIMODALITY_COEFF_TH = 0.555
MIN_BIMODAL_TESTS   = 1       # pass at least 1 out of 3 tests

# ── Plot settings ───────────────────────────────────────────
SAMPLES_PER_PAGE = 6
COLS_PER_PAGE    = 3
ROWS_PER_PAGE    = 2
DPI              = 100

print("✓ Configuration done")
print(f"  Expression : {EXPR_FILE}")
print(f"  Metadata   : {META_FILE}")
print(f"  Output dir : {OUT_DIR}")

✓ Configuration done
  Expression : /content/drive/MyDrive/Compendium_Dataset/compendium_expression.csv.gz
  Metadata   : /content/drive/MyDrive/Compendium_Dataset/compendium_metadata.csv
  Output dir : /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering


In [6]:
# ============================================================
# CELL 3: Inspect Raw Files — Know Before Loading
# ============================================================
import gzip
from pathlib import Path

def peek_file(path, n_lines=6):
    """Show first N raw lines of .gz or plain file"""
    path = str(path)
    name = Path(path).name
    print(f"\n{'='*60}")
    print(f"FILE: {name}")
    print(f"{'='*60}")
    try:
        if path.endswith(".gz"):
            with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f):
                    if i >= n_lines:
                        break
                    preview = line.rstrip()[:130]
                    print(f"  Row {i:>3}: {preview}")
        else:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f):
                    if i >= n_lines:
                        break
                    preview = line.rstrip()[:130]
                    print(f"  Row {i:>3}: {preview}")
    except Exception as e:
        print(f"  [ERROR] {e}")

# Inspect both files
peek_file(EXPR_FILE, n_lines=4)
peek_file(META_FILE, n_lines=4)


FILE: compendium_expression.csv.gz
  Row   0: gene_id,GSE152418_GPL24676_S145_nCOV001_C,GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1,GSE152418_GPL24676_S149_nCoV002EUHM-Draw-2,GS
  Row   1: ENSG00000000419,336,597,201,504,272,308,352,406,342,324,522,239,455,458,617,250,412,357,410,486,441,624,519,468,391,424,585,499,52
  Row   2: ENSG00000000457,352,334,238,242,168,246,345,188,185,284,395,194,246,344,437,234,280,366,448,498,400,590,426,522,482,514,429,372,32
  Row   3: ENSG00000000460,131,169,112,212,128,182,101,84,97,142,224,104,182,221,231,103,149,139,120,153,161,134,141,141,145,155,232,184,112,

FILE: compendium_metadata.csv
  Row   0: GSM,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,days_post_symptom_
  Row   1: GSM4614985,S145_nCOV001_C,GSM4614985,Public on Jul 31 2020,Jun 13 2020,Feb 01 2021,SRA,1,PBMC,Homo sapiens,40,M,Convalescent,Conva
  Row   2: GSM4614986,S147_nCoV001EUHM-Draw-1,GSM4614986,Public on Jul 31 2020,Ju

In [7]:
# ============================================================
# CELL 4: Load Data — Auto Format Detection
# ============================================================
import pandas as pd
import numpy as np
import gzip
from pathlib import Path

def detect_separator(path, n_lines=3):
    """Count tabs vs commas to detect separator"""
    path = str(path)
    lines = []
    try:
        if path.endswith(".gz"):
            with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f):
                    if i >= n_lines:
                        break
                    lines.append(line)
        else:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f):
                    if i >= n_lines:
                        break
                    lines.append(line)
    except Exception:
        return ","

    tabs   = sum(l.count("\t") for l in lines)
    commas = sum(l.count(",")  for l in lines)
    sep    = "\t" if tabs > commas else ","
    print(f"  → Separator detected: {'TAB' if sep == chr(9) else 'COMMA'} "
          f"(tabs={tabs}, commas={commas})")
    return sep


def load_dataframe(path, label="file", index_col=0):
    """
    Robustly load any CSV/TSV .gz or plain file.
    Tries multiple separators if first attempt fails.
    """
    path = str(path)
    print(f"\n[LOAD] {label}: {Path(path).name}")

    sep = detect_separator(path)
    compress = "gzip" if path.endswith(".gz") else None

    for try_sep in [sep, "\t", ","]:
        try:
            df = pd.read_csv(
                path,
                sep         = try_sep,
                index_col   = index_col,
                low_memory  = False,
                compression = compress,
                encoding    = "utf-8",
                on_bad_lines= "skip",
            )
            if df.shape[1] >= 1:
                sep_name = "TAB" if try_sep == "\t" else "COMMA"
                print(f"  ✓ Loaded with {sep_name} separator")
                print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]:,} cols")
                return df
        except Exception as e:
            print(f"  [WARN] Failed with sep={'TAB' if try_sep==chr(9) else 'COMMA'}: {e}")
            continue

    raise ValueError(f"[ERROR] Could not load: {path}")


# ── Load expression matrix ────────────────────────────────────
expr = load_dataframe(EXPR_FILE, label="Expression Matrix", index_col=0)

print(f"\n── Expression Details ──────────────────────────────")
print(f"  Genes (rows)       : {expr.shape[0]:,}")
print(f"  Samples (cols)     : {expr.shape[1]:,}")
print(f"  Index name         : {expr.index.name}")
print(f"  Index type sample  : {expr.index[:3].tolist()}")
print(f"  Column sample      : {expr.columns[:3].tolist()}")
print(f"  Data types         : {expr.dtypes.value_counts().to_dict()}")
print(f"  NaN total          : {expr.isnull().sum().sum():,}")
print(f"  Zero total         : {int((expr == 0).sum().sum()):,}")

print(f"\n── Expression Preview (4 genes × 4 samples) ────────")
print(expr.iloc[:4, :4])


# ── Load metadata ─────────────────────────────────────────────
meta = load_dataframe(META_FILE, label="Metadata", index_col=0)

print(f"\n── Metadata Details ────────────────────────────────")
print(f"  Samples (rows) : {meta.shape[0]:,}")
print(f"  Columns        : {meta.shape[1]:,}")
print(f"  Index name     : {meta.index.name}")
print(f"  Index sample   : {meta.index[:3].tolist()}")
print(f"  All columns    : {meta.columns.tolist()}")

print(f"\n── Metadata Preview (transposed) ───────────────────")
print(meta.head(3).T.iloc[:20])


# ── Alignment check ───────────────────────────────────────────
print(f"\n── Sample Alignment Check ──────────────────────────")
expr_cols = set(expr.columns.astype(str))
meta_idx  = set(meta.index.astype(str))

matched_direct = expr_cols & meta_idx
print(f"  Expression samples       : {len(expr_cols):,}")
print(f"  Metadata samples         : {len(meta_idx):,}")
print(f"  Matched (direct)         : {len(matched_direct):,}")

# Try via sample_id column
if "sample_id" in meta.columns:
    meta_sid  = set(meta["sample_id"].astype(str))
    sid_match = expr_cols & meta_sid
    print(f"  Matched via sample_id col: {len(sid_match):,}")
    if len(sid_match) > len(matched_direct):
        print("  [FIX] Re-indexing metadata by 'sample_id' column ...")
        meta = meta.set_index("sample_id")
        meta.index.name = "sample_id"
        matched_direct = set(expr.columns.astype(str)) & set(meta.index.astype(str))
        print(f"  Matched after fix        : {len(matched_direct):,}")

# Show mismatched examples
expr_only = sorted(list(expr_cols - set(meta.index.astype(str))))[:5]
meta_only = sorted(list(set(meta.index.astype(str)) - expr_cols))[:5]
print(f"  Expr-only (first 5): {expr_only}")
print(f"  Meta-only (first 5): {meta_only}")

print(f"\n✓ Data loading complete!")


[LOAD] Expression Matrix: compendium_expression.csv.gz
  → Separator detected: COMMA (tabs=0, commas=5511)
  ✓ Loaded with COMMA separator
  Shape : 27,416 rows × 1,837 cols

── Expression Details ──────────────────────────────
  Genes (rows)       : 27,416
  Samples (cols)     : 1,837
  Index name         : gene_id
  Index type sample  : ['ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460']
  Column sample      : ['GSE152418_GPL24676_S145_nCOV001_C', 'GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1', 'GSE152418_GPL24676_S149_nCoV002EUHM-Draw-2']
  Data types         : {dtype('int64'): 1837}
  NaN total          : 0
  Zero total         : 17,634,778

── Expression Preview (4 genes × 4 samples) ────────
                 GSE152418_GPL24676_S145_nCOV001_C  \
gene_id                                              
ENSG00000000419                                336   
ENSG00000000457                                352   
ENSG00000000460                                131   
ENSG00000000938   

In [8]:
# ============================================================
# CELL 4B: Fix Sample Alignment — Build ID Bridge
# ============================================================
import pandas as pd
import numpy as np
import re

print("=" * 60)
print(" SAMPLE ID ALIGNMENT FIX")
print("=" * 60)

print(f"\n  Expression cols  : {expr.shape[1]:,}  (format: GSE_GPL_SampleName)")
print(f"  Metadata rows    : {meta.shape[0]:,}  (format: GSMxxxxxxx)")
print(f"  Direct match     : 0  ← MISMATCH — fixing now...")

# ── Step 1: Extract GSE from expression column names ─────────
def extract_gse_from_col(col_name: str) -> str:
    """GSE152418_GPL24676_S145_nCOV001_C → GSE152418"""
    m = re.match(r"(GSE\d+)", col_name)
    return m.group(1) if m else "UNKNOWN"

# Build expression column info table
expr_col_df = pd.DataFrame({
    "expr_col" : expr.columns.tolist(),
    "GSE"      : [extract_gse_from_col(c) for c in expr.columns],
})
expr_col_df.index = expr_col_df["expr_col"]

print(f"\n── GSE breakdown in expression columns ──────────────")
for gse, grp in expr_col_df.groupby("GSE"):
    print(f"  {gse:<15}: {len(grp):>5} samples  | e.g.: {grp['expr_col'].iloc[0][:60]}")

# ── Step 2: Check metadata columns for title / sample name ──
print(f"\n── Metadata columns available for matching ──────────")
useful_cols = ["title", "geo_accession", "GSE", "GPL", "condition",
               "disease", "source_name_ch1"]
for c in useful_cols:
    if c in meta.columns:
        print(f"  '{c}' → sample: {meta[c].iloc[0]}")

 SAMPLE ID ALIGNMENT FIX

  Expression cols  : 1,837  (format: GSE_GPL_SampleName)
  Metadata rows    : 1,562  (format: GSMxxxxxxx)
  Direct match     : 0  ← MISMATCH — fixing now...

── GSE breakdown in expression columns ──────────────
  GSE107991      :    54 samples  | e.g.: GSE107991_GPL20301_Berry_London_Sample1
  GSE137317      :    74 samples  | e.g.: GSE137317_GPL17303_MF0015_1
  GSE140809      :   136 samples  | e.g.: GSE140809_GPL20301_1068x1q2
  GSE152418      :    34 samples  | e.g.: GSE152418_GPL24676_S145_nCOV001_C
  GSE153792      :    34 samples  | e.g.: GSE153792_GPL17303_13_1
  GSE161731      :   201 samples  | e.g.: GSE161731_GPL24676_94189
  GSE166190      :    98 samples  | e.g.: GSE166190_GPL20301_Adult17_Neg_Visit1
  GSE174482      :   134 samples  | e.g.: GSE174482_GPL24676_001_GS0613_A_DF_pos_400
  GSE178240      :   414 samples  | e.g.: GSE178240_GPL16791_1_GS0373_PBMC_Deng1
  GSE185263      :   392 samples  | e.g.: GSE185263_GPL16791_sepcol001
  GSE193978   

In [9]:
# ── Step 3: Build GSM → expr_col mapping via 'title' column ──
print("\n── Strategy: Match metadata 'title' → expression column suffix ──")

# Clean title strings for matching
def clean_for_match(s: str) -> str:
    """Lowercase, remove spaces/special chars for fuzzy match"""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

# Build lookup: cleaned_title → GSM
title_to_gsm = {}
if "title" in meta.columns:
    for gsm, row in meta.iterrows():
        cleaned = clean_for_match(row["title"])
        title_to_gsm[cleaned] = gsm

print(f"  Built title→GSM lookup: {len(title_to_gsm):,} entries")
print(f"  Example titles from metadata:")
for gsm in meta.index[:5]:
    t = meta.loc[gsm, "title"]
    print(f"    {gsm} → '{t}'")

print(f"\n  Example expr column suffixes (after GSE_GPL_ prefix):")
for col in expr.columns[:5]:
    parts = col.split("_", 2)  # split GSE, GPL, rest
    suffix = parts[2] if len(parts) > 2 else col
    print(f"    '{col}' → suffix='{suffix}'")


── Strategy: Match metadata 'title' → expression column suffix ──
  Built title→GSM lookup: 1,562 entries
  Example titles from metadata:
    GSM4614985 → 'S145_nCOV001_C'
    GSM4614986 → 'S147_nCoV001EUHM-Draw-1'
    GSM4614987 → 'S149_nCoV002EUHM-Draw-2'
    GSM4614988 → 'S150_nCoV003EUHM-Draw-1'
    GSM4614989 → 'S151_nCoV004EUHM-Draw-1'

  Example expr column suffixes (after GSE_GPL_ prefix):
    'GSE152418_GPL24676_S145_nCOV001_C' → suffix='S145_nCOV001_C'
    'GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1' → suffix='S147_nCoV001EUHM-Draw-1'
    'GSE152418_GPL24676_S149_nCoV002EUHM-Draw-2' → suffix='S149_nCoV002EUHM-Draw-2'
    'GSE152418_GPL24676_S150_nCoV003EUHM-Draw-1' → suffix='S150_nCoV003EUHM-Draw-1'
    'GSE152418_GPL24676_S151_nCoV004EUHM-Draw-1' → suffix='S151_nCoV004EUHM-Draw-1'


In [10]:
# ── Step 4: Multi-strategy mapping ───────────────────────────
print("\n── Building expr_col → GSM bridge ──────────────────────")

bridge = {}          # expr_col → GSM
unmatched_cols = []

for col in expr.columns:
    # Strategy A: suffix of expr col matches cleaned title
    parts = col.split("_", 2)
    suffix = parts[2] if len(parts) > 2 else col
    cleaned_suffix = clean_for_match(suffix)

    if cleaned_suffix in title_to_gsm:
        bridge[col] = title_to_gsm[cleaned_suffix]
        continue

    # Strategy B: full col cleaned matches any title
    cleaned_full = clean_for_match(col)
    if cleaned_full in title_to_gsm:
        bridge[col] = title_to_gsm[cleaned_full]
        continue

    # Strategy C: partial match — suffix starts with cleaned title
    matched = False
    for t_clean, gsm in title_to_gsm.items():
        if t_clean and (cleaned_suffix.startswith(t_clean) or
                        t_clean.startswith(cleaned_suffix)):
            bridge[col] = gsm
            matched = True
            break
    if matched:
        continue

    # Strategy D: match by GSE + position (fallback)
    unmatched_cols.append(col)

n_matched   = len(bridge)
n_unmatched = len(unmatched_cols)

print(f"\n  Matched via title  : {n_matched:,}")
print(f"  Unmatched          : {n_unmatched:,}")

if n_matched > 0:
    print(f"\n  Sample bridge entries (first 5):")
    for k, v in list(bridge.items())[:5]:
        title = meta.loc[v, "title"] if v in meta.index else "?"
        print(f"    {k[:55]:<55} → {v}  (title='{title}')")


── Building expr_col → GSM bridge ──────────────────────

  Matched via title  : 845
  Unmatched          : 992

  Sample bridge entries (first 5):
    GSE152418_GPL24676_S145_nCOV001_C                       → GSM4614985  (title='S145_nCOV001_C')
    GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1              → GSM4614986  (title='S147_nCoV001EUHM-Draw-1')
    GSE152418_GPL24676_S149_nCoV002EUHM-Draw-2              → GSM4614987  (title='S149_nCoV002EUHM-Draw-2')
    GSE152418_GPL24676_S150_nCoV003EUHM-Draw-1              → GSM4614988  (title='S150_nCoV003EUHM-Draw-1')
    GSE152418_GPL24676_S151_nCoV004EUHM-Draw-1              → GSM4614989  (title='S151_nCoV004EUHM-Draw-1')


In [11]:
# ── Step 5: Fallback — match by GSE + row order ──────────────
if n_unmatched > 0:
    print(f"\n── Fallback: Matching {n_unmatched:,} unmatched cols by GSE+order ──")

    # Group unmatched by GSE
    from collections import defaultdict
    unmatched_by_gse = defaultdict(list)
    for col in unmatched_cols:
        gse = extract_gse_from_col(col)
        unmatched_by_gse[gse].append(col)

    # Group metadata by GSE
    meta_by_gse = defaultdict(list)
    if "GSE" in meta.columns:
        for gsm, row in meta.iterrows():
            gse = str(row.get("GSE", "")).strip()
            if gse:
                meta_by_gse[gse].append(gsm)

    fallback_matched = 0
    still_unmatched  = []

    for gse, cols_in_gse in unmatched_by_gse.items():
        gsms_in_gse = meta_by_gse.get(gse, [])

        # Filter out already-bridged GSMs
        used_gsms = set(bridge.values())
        available_gsms = [g for g in gsms_in_gse if g not in used_gsms]

        for i, col in enumerate(cols_in_gse):
            if i < len(available_gsms):
                bridge[col] = available_gsms[i]
                fallback_matched += 1
            else:
                still_unmatched.append(col)

    print(f"  Matched by GSE+order : {fallback_matched:,}")
    print(f"  Still unmatched      : {len(still_unmatched):,}")

    if still_unmatched:
        print(f"  Still unmatched examples:")
        for col in still_unmatched[:5]:
            print(f"    {col}")


── Fallback: Matching 992 unmatched cols by GSE+order ──
  Matched by GSE+order : 769
  Still unmatched      : 223
  Still unmatched examples:
    GSE161731_GPL24676_DU09-02S0000150_batch2
    GSE161731_GPL24676_DU09-02S0000154_batch2
    GSE161731_GPL24676_DU09-02S0000158_batch2
    GSE178240_GPL16791_376_1_0288_CD3
    GSE178240_GPL16791_377_1_0288_CD4


In [12]:
# ── Step 6: Build aligned metadata ───────────────────────────
print("\n── Building aligned metadata table ─────────────────────")

# Create bridge DataFrame
bridge_df = pd.DataFrame([
    {"expr_col": col, "GSM": gsm}
    for col, gsm in bridge.items()
])

# Add metadata columns to bridge
if not bridge_df.empty:
    meta_bridged = bridge_df.set_index("GSM").join(meta, how="left")
    meta_bridged.index = bridge_df["expr_col"].values
    meta_bridged.index.name = "sample_id"
    meta_bridged.insert(0, "GSM", bridge_df["GSM"].values)
else:
    meta_bridged = pd.DataFrame()

print(f"  Bridge table shape : {bridge_df.shape}")
print(f"  Aligned meta shape : {meta_bridged.shape}")

# Check alignment
aligned_samples = set(bridge.keys()) & set(expr.columns.astype(str))
print(f"  Final aligned samples: {len(aligned_samples):,} / {expr.shape[1]:,}")

print(f"\n  Sample of aligned metadata:")
print(meta_bridged.iloc[:3][["GSM","disease","GSE",
                              "title","condition","severity"]
                              if all(c in meta_bridged.columns
                              for c in ["disease","GSE","title","condition","severity"])
                              else meta_bridged.columns[:6]].T)


── Building aligned metadata table ─────────────────────
  Bridge table shape : (1614, 2)
  Aligned meta shape : (1614, 82)
  Final aligned samples: 1,614 / 1,837

  Sample of aligned metadata:
sample_id GSE152418_GPL24676_S145_nCOV001_C  \
GSM                              GSM4614985   
disease                            Covid-19   
GSE                               GSE152418   
title                        S145_nCOV001_C   
condition             COVID-19_convalescent   
severity                       Convalescent   

sample_id GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1  \
GSM                                       GSM4614986   
disease                                     Covid-19   
GSE                                        GSE152418   
title                        S147_nCoV001EUHM-Draw-1   
condition                          COVID-19_moderate   
severity                                    Moderate   

sample_id GSE152418_GPL24676_S149_nCoV002EUHM-Draw-2  
GSM                        

In [13]:
# ── Step 7: Reindex expression with GSM IDs (optional) ────────
print("\n── Relabeling expression columns → GSM IDs ─────────────")

# Map expr cols → GSM (only for matched ones)
matched_cols = [c for c in expr.columns if c in bridge]
gsm_labels   = [bridge[c] for c in matched_cols]

expr_aligned = expr[matched_cols].copy()
expr_aligned.columns = gsm_labels

print(f"  Expression before relabel : {expr.shape}")
print(f"  Expression after relabel  : {expr_aligned.shape}")
print(f"  New column sample         : {expr_aligned.columns[:4].tolist()}")

# Verify alignment
meta_aligned = meta.loc[meta.index.isin(gsm_labels)].copy()
final_match  = set(expr_aligned.columns) & set(meta_aligned.index)
print(f"\n  ✓ Final matched samples: {len(final_match):,}")

# Update meta with disease column check
print(f"\n── Disease distribution in aligned metadata ─────────────")
if "disease" in meta_aligned.columns:
    print(meta_aligned["disease"].value_counts().to_string())
elif "disease state" in meta_aligned.columns:
    print(meta_aligned["disease state"].value_counts().to_string())


── Relabeling expression columns → GSM IDs ─────────────
  Expression before relabel : (27416, 1837)
  Expression after relabel  : (27416, 1614)
  New column sample         : ['GSM4614985', 'GSM4614986', 'GSM4614987', 'GSM4614988']

  ✓ Final matched samples: 1,472

── Disease distribution in aligned metadata ─────────────
disease
Spesis      392
Diabetes    374
Covid-19    330
Dengue      322
TB           54


In [14]:
# ── Step 8: Save bridge + aligned files ──────────────────────
print("\n── Saving bridge and aligned files ──────────────────────")

# Save bridge table
bridge_out = OUT_DIR / "sample_id_bridge.tsv"
bridge_df.to_csv(bridge_out, sep="\t", index=False)
print(f"  ✓ Bridge table    : {bridge_out}")

# Save aligned metadata (with expr_col as index)
meta_bridge_out = OUT_DIR / "metadata_aligned.tsv"
meta_bridged.to_csv(meta_bridge_out, sep="\t", index=True)
print(f"  ✓ Aligned metadata: {meta_bridge_out}")

# Save aligned expression (with GSM as columns)
expr_aligned_out = OUT_DIR / "expression_aligned.tsv.gz"
expr_aligned.to_csv(expr_aligned_out, sep="\t", index=True, compression="gzip")
print(f"  ✓ Aligned expr    : {expr_aligned_out}")

# Update variables for downstream cells
meta = meta_aligned
expr_log_raw = expr_aligned.copy()

print(f"\n✓ Alignment complete!")
print(f"  Expression : {expr_aligned.shape[0]:,} genes × {expr_aligned.shape[1]:,} samples")
print(f"  Metadata   : {meta_aligned.shape[0]:,} samples × {meta_aligned.shape[1]:,} columns")


── Saving bridge and aligned files ──────────────────────
  ✓ Bridge table    : /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/sample_id_bridge.tsv
  ✓ Aligned metadata: /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/metadata_aligned.tsv
  ✓ Aligned expr    : /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/expression_aligned.tsv.gz

✓ Alignment complete!
  Expression : 27,416 genes × 1,614 samples
  Metadata   : 1,472 samples × 80 columns


In [15]:
# ── Step 9: Now apply log1p on aligned expression ─────────────
print("\n── Applying log1p to aligned expression ─────────────────")

expr_aligned_num = expr_aligned.apply(pd.to_numeric, errors="coerce").fillna(0)
expr_log = np.log1p(expr_aligned_num)

print(f"  Shape  : {expr_log.shape[0]:,} genes × {expr_log.shape[1]:,} samples")
print(f"  Min    : {expr_log.min().min():.4f}")
print(f"  Max    : {expr_log.max().max():.4f}")
print(f"  Mean   : {expr_log.mean().mean():.4f}")

print("\n✓ Ready for bimodal testing!")
print("   → Run CELL 6 (Bimodal Functions) next")
print("   → Then CELL 7 (Run Tests)")


── Applying log1p to aligned expression ─────────────────
  Shape  : 27,416 genes × 1,614 samples
  Min    : 0.0000
  Max    : 14.9669
  Mean   : 2.8766

✓ Ready for bimodal testing!
   → Run CELL 6 (Bimodal Functions) next
   → Then CELL 7 (Run Tests)


In [16]:
# ============================================================
# CELL 5: Log1p Transformation
# ============================================================
print("[TRANSFORM] Converting all columns to numeric ...")
expr = expr.apply(pd.to_numeric, errors="coerce")

print("[TRANSFORM] Filling NaN → 0 (not detected = 0 counts) ...")
n_nan = expr.isnull().sum().sum()
print(f"  NaN values filled : {n_nan:,}")
expr = expr.fillna(0)

print("[TRANSFORM] Applying log1p ...")
expr_log = np.log1p(expr)

# Sanity check
print(f"\n── After log1p ─────────────────────────────────────")
print(f"  Shape  : {expr_log.shape[0]:,} genes × {expr_log.shape[1]:,} samples")
print(f"  Min    : {expr_log.min().min():.4f}")
print(f"  Max    : {expr_log.max().max():.4f}")
print(f"  Mean   : {expr_log.mean().mean():.4f}")
print(f"  NaN    : {expr_log.isnull().sum().sum():,}")

# Quick per-sample stats
sample_stats = pd.DataFrame({
    "mean"    : expr_log.mean(),
    "std"     : expr_log.std(),
    "min"     : expr_log.min(),
    "max"     : expr_log.max(),
    "n_zeros" : (expr_log == 0).sum(),
})
print(f"\n── Per-sample statistics (first 5) ─────────────────")
print(sample_stats.head())
print(f"\n── Per-sample statistics (overall distribution) ────")
print(sample_stats.describe().round(3))

print("\n✓ Log1p transform complete")

[TRANSFORM] Converting all columns to numeric ...
[TRANSFORM] Filling NaN → 0 (not detected = 0 counts) ...
  NaN values filled : 0
[TRANSFORM] Applying log1p ...

── After log1p ─────────────────────────────────────
  Shape  : 27,416 genes × 1,837 samples
  Min    : 0.0000
  Max    : 14.9669
  Mean   : 2.8278
  NaN    : 0

── Per-sample statistics (first 5) ─────────────────
                                                mean       std  min  \
GSE152418_GPL24676_S145_nCOV001_C           3.272181  2.723160  0.0   
GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1  3.319671  2.638336  0.0   
GSE152418_GPL24676_S149_nCoV002EUHM-Draw-2  3.216976  2.631224  0.0   
GSE152418_GPL24676_S150_nCoV003EUHM-Draw-1  3.386868  2.807462  0.0   
GSE152418_GPL24676_S151_nCoV004EUHM-Draw-1  3.127429  2.712430  0.0   

                                                  max  n_zeros  
GSE152418_GPL24676_S145_nCOV001_C           12.691790     5361  
GSE152418_GPL24676_S147_nCoV001EUHM-Draw-1  12.565612     4450  

In [17]:
# ============================================================
# CELL 6: Bimodal Detection Functions
# ============================================================
import diptest
from scipy import stats
from scipy.stats import skew, kurtosis
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings("ignore")


def hartigan_dip_test(values: np.ndarray):
    """
    Hartigan's Dip Test.
    Small p-value (< alpha) → bimodal
    """
    clean = values[np.isfinite(values) & (values > 0)]
    if len(clean) < 10:
        return np.nan, np.nan
    try:
        dip, pval = diptest.diptest(clean)
        return float(dip), float(pval)
    except Exception:
        return np.nan, np.nan


def bimodality_coefficient(values: np.ndarray) -> float:
    """
    BC = (skew² + 1) / (kurtosis + correction)
    BC > 0.555 → bimodal
    """
    clean = values[np.isfinite(values) & (values > 0)]
    n = len(clean)
    if n < 4:
        return np.nan
    try:
        sk  = skew(clean)
        ku  = kurtosis(clean)
        adj = 3 * (n - 1)**2 / ((n - 2) * (n - 3))
        bc  = (sk**2 + 1) / (ku + adj)
        return float(bc)
    except Exception:
        return np.nan


def bimodality_index(values: np.ndarray) -> float:
    """
    Bimodality Index via 2-component GMM.
    BI > 1.1 → bimodal
    """
    clean = values[np.isfinite(values) & (values > 0)].reshape(-1, 1)
    if len(clean) < 20:
        return np.nan
    try:
        gm = GaussianMixture(
            n_components = 2,
            random_state = 42,
            max_iter     = 300,
            n_init       = 3,
        )
        gm.fit(clean)
        means   = gm.means_.flatten()
        stds    = np.sqrt(gm.covariances_.flatten())
        weights = gm.weights_.flatten()
        delta   = abs(means[0] - means[1]) / np.sqrt(stds[0]**2 + stds[1]**2 + 1e-9)
        bi      = delta * 2 * weights[0] * weights[1] * 2
        return float(bi)
    except Exception:
        return np.nan


def classify_sample(values: np.ndarray) -> dict:
    """
    Run all 3 bimodality tests on one sample.
    Returns dict with statistics + BIMODAL verdict.
    """
    clean = values[np.isfinite(values) & (values > 0)]

    dip, dip_pval = hartigan_dip_test(values)
    bc            = bimodality_coefficient(values)
    bi            = bimodality_index(values)

    pass_dip = bool(dip_pval < HARTIGAN_DIP_ALPHA)  if not np.isnan(dip_pval) else False
    pass_bc  = bool(bc  > BIMODALITY_COEFF_TH)      if not np.isnan(bc)       else False
    pass_bi  = bool(bi  > 1.1)                       if not np.isnan(bi)       else False

    n_passed = int(pass_dip) + int(pass_bc) + int(pass_bi)

    return {
        "n_expressed"       : int((values > 0).sum()),
        "mean"              : float(np.mean(clean))   if len(clean) > 0 else np.nan,
        "std"               : float(np.std(clean))    if len(clean) > 0 else np.nan,
        "skewness"          : float(skew(clean))      if len(clean) > 3 else np.nan,
        "kurtosis"          : float(kurtosis(clean))  if len(clean) > 3 else np.nan,
        "dip_stat"          : dip,
        "dip_pval"          : dip_pval,
        "pass_dip"          : pass_dip,
        "bimodality_coeff"  : bc,
        "pass_bc"           : pass_bc,
        "bimodality_index"  : bi,
        "pass_bi"           : pass_bi,
        "n_tests_passed"    : n_passed,
        "is_bimodal"        : n_passed >= MIN_BIMODAL_TESTS,
    }


print("✓ Bimodal detection functions ready")
print(f"  Tests:")
print(f"    1. Hartigan Dip Test  → p < {HARTIGAN_DIP_ALPHA}")
print(f"    2. Bimodality Coeff   → BC > {BIMODALITY_COEFF_TH}")
print(f"    3. Bimodality Index   → BI > 1.1")
print(f"  A sample is BIMODAL if it passes ≥ {MIN_BIMODAL_TESTS} test(s)")

✓ Bimodal detection functions ready
  Tests:
    1. Hartigan Dip Test  → p < 0.05
    2. Bimodality Coeff   → BC > 0.555
    3. Bimodality Index   → BI > 1.1
  A sample is BIMODAL if it passes ≥ 1 test(s)


In [18]:
# ============================================================
# CELL 7: Run Bimodal Tests on ALL Samples
# ============================================================
from tqdm.notebook import tqdm

n_samples = expr_log.shape[1]
print(f"[TEST] Running bimodal tests on {n_samples:,} samples ...")

results = {}
for sample_id in tqdm(expr_log.columns, desc="Bimodal Testing", unit="sample"):
    values = expr_log[sample_id].values.astype(float)
    results[sample_id] = classify_sample(values)

# Build results table
results_df = pd.DataFrame(results).T
results_df.index.name = "sample_id"

# Convert bool columns
for col in ["pass_dip", "pass_bc", "pass_bi", "is_bimodal"]:
    results_df[col] = results_df[col].astype(bool)

# Summary
n_total   = len(results_df)
n_bimodal = int(results_df["is_bimodal"].sum())
n_reject  = n_total - n_bimodal

print(f"\n{'='*55}")
print(f"  BIMODAL FILTERING RESULTS")
print(f"{'='*55}")
print(f"  Total samples tested    : {n_total:,}")
print(f"  ✓ Bimodal  (KEEP)       : {n_bimodal:,}  ({100*n_bimodal/n_total:.1f}%)")
print(f"  ✗ Unimodal (REMOVE)     : {n_reject:,}  ({100*n_reject/n_total:.1f}%)")
print(f"\n  Individual test pass rates:")
print(f"    Dip test (p<{HARTIGAN_DIP_ALPHA})  : {results_df['pass_dip'].sum():,} samples")
print(f"    BC  test (BC>{BIMODALITY_COEFF_TH}) : {results_df['pass_bc'].sum():,} samples")
print(f"    BI  test (BI>1.1)     : {results_df['pass_bi'].sum():,} samples")

print(f"\n── Test Statistics Summary ──")
print(results_df[["dip_pval","bimodality_coeff","bimodality_index","n_tests_passed"]].describe().round(4))

# Save test results
res_out = OUT_DIR / "bimodal_test_results.tsv"
results_df.to_csv(res_out, sep="\t", index=True)
print(f"\n✓ Results saved: {res_out}")

[TEST] Running bimodal tests on 1,837 samples ...


Bimodal Testing:   0%|          | 0/1837 [00:00<?, ?sample/s]


  BIMODAL FILTERING RESULTS
  Total samples tested    : 1,837
  ✓ Bimodal  (KEEP)       : 1,837  (100.0%)
  ✗ Unimodal (REMOVE)     : 0  (0.0%)

  Individual test pass rates:
    Dip test (p<0.05)  : 1,837 samples
    BC  test (BC>0.555) : 62 samples
    BI  test (BI>1.1)     : 1,806 samples

── Test Statistics Summary ──
        dip_pval  bimodality_coeff  bimodality_index  n_tests_passed
count     1837.0         1837.0000         1837.0000            1837
unique      59.0         1837.0000         1837.0000               3
top          0.0            0.5049            2.1855               2
freq      1779.0            1.0000            1.0000            1744

✓ Results saved: /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/bimodal_test_results.tsv


In [19]:
# ============================================================
# CELL 8: Filter Expression + Metadata → Save
# ============================================================

bimodal_samples  = results_df[results_df["is_bimodal"]].index.tolist()
unimodal_samples = results_df[~results_df["is_bimodal"]].index.tolist()

print(f"[FILTER] Bimodal samples  : {len(bimodal_samples):,}")
print(f"[FILTER] Unimodal samples : {len(unimodal_samples):,}")

# ── Filter expression ──────────────────────────────────────
expr_bimodal = expr_log[bimodal_samples]
print(f"\n[EXPR] Filtered shape: {expr_bimodal.shape[0]:,} genes × {expr_bimodal.shape[1]:,} samples")

# ── Filter metadata ────────────────────────────────────────
meta_idx_set = set(meta.index.astype(str))
meta_sid_set = set(meta["sample_id"].astype(str)) if "sample_id" in meta.columns else set()

if meta_idx_set & set(bimodal_samples):
    meta_bimodal = meta.loc[meta.index.isin(bimodal_samples)].copy()
elif meta_sid_set & set(bimodal_samples):
    meta_bimodal = meta[meta["sample_id"].isin(bimodal_samples)].copy()
else:
    print("[WARN] Could not align metadata with bimodal samples — keeping all metadata")
    meta_bimodal = meta.copy()

# Attach test scores to metadata
meta_bimodal = meta_bimodal.join(
    results_df[[
        "dip_pval", "bimodality_coeff",
        "bimodality_index", "n_tests_passed", "is_bimodal"
    ]],
    how="left",
)

print(f"[META] Filtered shape: {meta_bimodal.shape[0]:,} samples × {meta_bimodal.shape[1]:,} cols")

# ── Disease breakdown ──────────────────────────────────────
if "disease" in meta_bimodal.columns:
    print(f"\n── Disease breakdown (bimodal samples) ─────────────")
    for disease, grp in meta_bimodal.groupby("disease"):
        n = len(grp)
        print(f"  {disease:<15}: {n:>5} samples")

# ── Save ───────────────────────────────────────────────────
expr_out = OUT_DIR / "compendium_expression_bimodal.tsv.gz"
meta_out = OUT_DIR / "compendium_metadata_bimodal.tsv"
removed_out = OUT_DIR / "removed_samples.tsv"

expr_bimodal.to_csv(expr_out, sep="\t", index=True, compression="gzip")
meta_bimodal.to_csv(meta_out, sep="\t", index=True)

# Save removed sample list
removed_df = results_df[~results_df["is_bimodal"]]
removed_df.to_csv(removed_out, sep="\t", index=True)

print(f"\n✓ Saved: {expr_out}")
print(f"✓ Saved: {meta_out}")
print(f"✓ Saved: {removed_out}")

[FILTER] Bimodal samples  : 1,837
[FILTER] Unimodal samples : 0

[EXPR] Filtered shape: 27,416 genes × 1,837 samples
[WARN] Could not align metadata with bimodal samples — keeping all metadata
[META] Filtered shape: 1,472 samples × 85 cols

── Disease breakdown (bimodal samples) ─────────────
  Covid-19       :   330 samples
  Dengue         :   322 samples
  Diabetes       :   374 samples
  Spesis         :   392 samples
  TB             :    54 samples

✓ Saved: /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/compendium_expression_bimodal.tsv.gz
✓ Saved: /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/compendium_metadata_bimodal.tsv
✓ Saved: /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/removed_samples.tsv


In [20]:
# ============================================================
# CELL 9: Generate PDF with All Sample Distributions
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as pdf_backend
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")


def plot_one_sample(ax, sample_id, values, result):
    """Plot histogram + KDE + GMM for one sample"""
    clean = values[np.isfinite(values) & (values > 0)]
    is_bm = result["is_bimodal"]
    color = "#2ecc71" if is_bm else "#e74c3c"
    label = "✓ BIMODAL" if is_bm else "✗ UNIMODAL"

    if len(clean) < 10:
        ax.text(0.5, 0.5, "Insufficient data",
                ha="center", va="center", transform=ax.transAxes, fontsize=9)
        ax.set_title(sample_id[:40], fontsize=7)
        return

    # Histogram
    ax.hist(clean, bins=60, density=True, alpha=0.35,
            color=color, edgecolor="none")

    # KDE
    try:
        kde = gaussian_kde(clean, bw_method="scott")
        xr  = np.linspace(clean.min(), clean.max(), 300)
        ax.plot(xr, kde(xr), "k-", lw=1.5, label="KDE")
    except Exception:
        pass

    # GMM 2-component
    try:
        gm = GaussianMixture(n_components=2, random_state=42, max_iter=300)
        gm.fit(clean.reshape(-1, 1))
        means   = gm.means_.flatten()
        stds    = np.sqrt(gm.covariances_.flatten())
        weights = gm.weights_.flatten()
        xr      = np.linspace(clean.min(), clean.max(), 300)
        gmm_sum = np.zeros_like(xr)
        for i, c in enumerate(["#e74c3c", "#3498db"]):
            comp = weights[i] * stats.norm.pdf(xr, means[i], stds[i])
            ax.plot(xr, comp, "--", color=c, lw=1.1, alpha=0.8, label=f"GMM-{i+1}")
            gmm_sum += comp
        ax.plot(xr, gmm_sum, "m-", lw=1.3, alpha=0.6, label="GMM total")
    except Exception:
        pass

    # Stats annotation
    annot = (
        f"{label}\n"
        f"Dip p ={result['dip_pval']:.3f}  {'✓' if result['pass_dip'] else '✗'}\n"
        f"BC    ={result['bimodality_coeff']:.3f}  {'✓' if result['pass_bc'] else '✗'}\n"
        f"BI    ={result['bimodality_index']:.3f}  {'✓' if result['pass_bi'] else '✗'}\n"
        f"Passed: {result['n_tests_passed']}/3"
    )
    ax.text(
        0.97, 0.97, annot,
        transform=ax.transAxes, fontsize=6,
        va="top", ha="right",
        bbox=dict(
            boxstyle    = "round,pad=0.3",
            facecolor   = "#d5f5e3" if is_bm else "#fadbd8",
            edgecolor   = color,
            alpha       = 0.9,
            linewidth   = 1.2,
        ),
    )

    short = sample_id if len(sample_id) <= 38 else sample_id[:35] + "..."
    ax.set_title(short, fontsize=7, pad=2)
    ax.set_xlabel("log1p(count)", fontsize=6)
    ax.set_ylabel("Density",      fontsize=6)
    ax.tick_params(labelsize=5)
    ax.legend(fontsize=4.5, loc="upper left", framealpha=0.7)


# ── Build PDF ─────────────────────────────────────────────────
pdf_path   = OUT_DIR / "sample_distributions_ALL.pdf"
sample_list = expr_log.columns.tolist()
n_pages    = int(np.ceil(len(sample_list) / SAMPLES_PER_PAGE))

print(f"[PDF] Building PDF ...")
print(f"      Samples  : {len(sample_list):,}")
print(f"      Pages    : {n_pages + 2} (cover + overview + {n_pages} sample pages)")

with pdf_backend.PdfPages(pdf_path) as pdf:

    # ── Cover page ────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis("off")

    disease_lines = ""
    if "disease" in meta_bimodal.columns:
        for d, grp in meta_bimodal.groupby("disease"):
            disease_lines += f"    {d:<15}: {len(grp):>5} bimodal samples\n"

    cover_txt = (
        "MULTI-DISEASE COMPENDIUM\n"
        "BIMODAL SAMPLE FILTERING REPORT\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
        f"  Total samples tested    : {n_total:>6,}\n"
        f"  ✓  Bimodal  (KEPT)      : {n_bimodal:>6,}   ({100*n_bimodal/n_total:.1f}%)\n"
        f"  ✗  Unimodal (REMOVED)   : {n_reject:>6,}   ({100*n_reject/n_total:.1f}%)\n\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "  Tests Applied:\n"
        f"    1. Hartigan Dip Test  →  p < {HARTIGAN_DIP_ALPHA}\n"
        f"    2. Bimodality Coefficient (BC) > {BIMODALITY_COEFF_TH}\n"
        f"    3. Bimodality Index (BI) > 1.1\n"
        f"    Pass criterion: ≥ {MIN_BIMODAL_TESTS} test(s) out of 3\n\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "  Disease Breakdown (Bimodal Samples):\n"
        f"{disease_lines}"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "  Transform : log1p(raw counts)\n"
        "  Green box : BIMODAL   (kept)\n"
        "  Red box   : UNIMODAL  (removed)\n"
    )
    ax.text(
        0.5, 0.5, cover_txt,
        transform=ax.transAxes,
        ha="center", va="center",
        fontsize=13, family="monospace",
        bbox=dict(boxstyle="round,pad=1",
                  facecolor="#eaf2ff",
                  edgecolor="#2980b9", lw=2),
    )
    ax.set_title("BIMODAL DISTRIBUTION FILTERING", fontsize=18,
                 fontweight="bold", pad=25)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # ── Overview stats page ───────────────────────────────
    fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
    fig2.suptitle("Overview Statistics", fontsize=14, fontweight="bold")

    # Pie
    axes2[0,0].pie(
        [n_bimodal, n_reject],
        labels=[f"Bimodal\n(n={n_bimodal})", f"Unimodal\n(n={n_reject})"],
        colors=["#2ecc71","#e74c3c"],
        autopct="%1.1f%%", startangle=90,
    )
    axes2[0,0].set_title("Sample Classification", fontweight="bold")

    # Dip p-value
    axes2[0,1].hist(results_df["dip_pval"].dropna(), bins=50,
                    color="#3498db", edgecolor="white", alpha=0.8)
    axes2[0,1].axvline(HARTIGAN_DIP_ALPHA, color="red", ls="--",
                       lw=2, label=f"α={HARTIGAN_DIP_ALPHA}")
    axes2[0,1].set_title("Dip Test p-values")
    axes2[0,1].set_xlabel("p-value"); axes2[0,1].set_ylabel("Samples")
    axes2[0,1].legend()

    # BC
    axes2[0,2].hist(results_df["bimodality_coeff"].dropna(), bins=50,
                    color="#9b59b6", edgecolor="white", alpha=0.8)
    axes2[0,2].axvline(BIMODALITY_COEFF_TH, color="red", ls="--",
                       lw=2, label=f"BC={BIMODALITY_COEFF_TH}")
    axes2[0,2].set_title("Bimodality Coefficient")
    axes2[0,2].set_xlabel("BC"); axes2[0,2].set_ylabel("Samples")
    axes2[0,2].legend()

    # BI
    axes2[1,0].hist(results_df["bimodality_index"].dropna(), bins=50,
                    color="#e67e22", edgecolor="white", alpha=0.8)
    axes2[1,0].axvline(1.1, color="red", ls="--", lw=2, label="BI=1.1")
    axes2[1,0].set_title("Bimodality Index")
    axes2[1,0].set_xlabel("BI"); axes2[1,0].set_ylabel("Samples")
    axes2[1,0].legend()

    # Tests passed
    cp = results_df["n_tests_passed"].value_counts().sort_index()
    bars = axes2[1,1].bar(
        cp.index, cp.values,
        color=["#e74c3c","#f39c12","#2ecc71","#27ae60"][:len(cp)],
        edgecolor="white",
    )
    axes2[1,1].set_title("Tests Passed per Sample")
    axes2[1,1].set_xlabel("N tests passed"); axes2[1,1].set_ylabel("Samples")
    axes2[1,1].set_xticks([0,1,2,3])
    for b in bars:
        axes2[1,1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
                        str(int(b.get_height())), ha="center", fontsize=9)

    # Disease breakdown
    if "disease" in meta.columns:
        d_total  = meta["disease"].value_counts()
        d_bim    = meta.loc[meta.index.isin(bimodal_samples), "disease"].value_counts()
        dlist    = d_total.index.tolist()
        xp       = np.arange(len(dlist))
        axes2[1,2].bar(xp - 0.2, [d_total.get(d,0) for d in dlist],
                       0.4, label="Total",   color="#95a5a6", edgecolor="white")
        axes2[1,2].bar(xp + 0.2, [d_bim.get(d,0)   for d in dlist],
                       0.4, label="Bimodal", color="#2ecc71", edgecolor="white")
        axes2[1,2].set_xticks(xp)
        axes2[1,2].set_xticklabels(dlist, rotation=30, ha="right")
        axes2[1,2].set_title("Bimodal vs Total per Disease")
        axes2[1,2].legend()

    plt.tight_layout()
    pdf.savefig(fig2, bbox_inches="tight")
    plt.close(fig2)

    # ── Per-sample pages ──────────────────────────────────
    for page_idx in range(n_pages):
        start = page_idx * SAMPLES_PER_PAGE
        end   = min(start + SAMPLES_PER_PAGE, len(sample_list))
        page_samples = sample_list[start:end]

        fig3, axes3 = plt.subplots(
            ROWS_PER_PAGE, COLS_PER_PAGE,
            figsize=(18, 10),
            constrained_layout=True,
        )
        flat = axes3.flatten()

        for i, sid in enumerate(page_samples):
            vals = expr_log[sid].values.astype(float)
            plot_one_sample(flat[i], sid, vals, results[sid])

        for j in range(len(page_samples), len(flat)):
            flat[j].axis("off")

        fig3.suptitle(
            f"Sample Distributions | Page {page_idx+1}/{n_pages} | "
            f"Samples {start+1}–{end} of {len(sample_list)}",
            fontsize=9, fontweight="bold",
        )
        pdf.savefig(fig3, bbox_inches="tight")
        plt.close(fig3)

        if (page_idx + 1) % 10 == 0:
            print(f"  ... {page_idx+1}/{n_pages} pages done")

print(f"\n✓ PDF saved: {pdf_path}")

[PDF] Building PDF ...
      Samples  : 1,837
      Pages    : 309 (cover + overview + 307 sample pages)
  ... 10/307 pages done
  ... 20/307 pages done
  ... 30/307 pages done
  ... 40/307 pages done
  ... 50/307 pages done
  ... 60/307 pages done
  ... 70/307 pages done
  ... 80/307 pages done
  ... 90/307 pages done
  ... 100/307 pages done
  ... 110/307 pages done
  ... 120/307 pages done
  ... 130/307 pages done
  ... 140/307 pages done
  ... 150/307 pages done
  ... 160/307 pages done
  ... 170/307 pages done
  ... 180/307 pages done
  ... 190/307 pages done
  ... 200/307 pages done
  ... 210/307 pages done
  ... 220/307 pages done
  ... 230/307 pages done
  ... 240/307 pages done
  ... 250/307 pages done
  ... 260/307 pages done
  ... 270/307 pages done
  ... 280/307 pages done
  ... 290/307 pages done
  ... 300/307 pages done

✓ PDF saved: /content/drive/MyDrive/Compendium_Dataset/Bimodal_Filtering/sample_distributions_ALL.pdf


In [21]:
# ============================================================
# CELL 10: Final Summary
# ============================================================
print("\n" + "="*60)
print("  FINAL SUMMARY")
print("="*60)
print(f"  Input  : {expr.shape[0]:,} genes × {expr.shape[1]:,} samples")
print(f"  Output : {expr_bimodal.shape[0]:,} genes × {expr_bimodal.shape[1]:,} samples")
print(f"\n  Bimodal (KEPT)    : {n_bimodal:,}  ({100*n_bimodal/n_total:.1f}%)")
print(f"  Unimodal (REMOVED): {n_reject:,}  ({100*n_reject/n_total:.1f}%)")

print(f"\n── Output Files ────────────────────────────────────")
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        mb = f.stat().st_size / 1e6
        print(f"  {f.name:<55} {mb:6.1f} MB")

print("\n✓ ALL DONE!")


  FINAL SUMMARY
  Input  : 27,416 genes × 1,837 samples
  Output : 27,416 genes × 1,837 samples

  Bimodal (KEPT)    : 1,837  (100.0%)
  Unimodal (REMOVED): 0  (0.0%)

── Output Files ────────────────────────────────────
  bimodal_test_results.tsv                                   0.4 MB
  compendium_expression_bimodal.tsv.gz                     131.1 MB
  compendium_metadata_bimodal.tsv                            1.4 MB
  expression_aligned.tsv.gz                                 44.2 MB
  metadata_aligned.tsv                                       1.6 MB
  removed_samples.tsv                                        0.0 MB
  sample_distributions_ALL.pdf                              10.7 MB
  sample_id_bridge.tsv                                       0.1 MB

✓ ALL DONE!
